# Lab 1.4 - Generate Images with a Diffusion Pipeline (Start)
**Day 1 - Session 4**

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/kpassoubady/gen-ai-companion-intuite/blob/main/day1/lab1_4_diffusion/start/lab1_4_diffusion.ipynb)

**Goal:** Run a pre-trained diffusion model (inference only) and see how it
turns random noise into images by denoising, one step at a time.

Look for the `# TODO` markers and fill them in. The notebook in the `solution/`
folder has the full answer if you get stuck.

The core lab uses a small unconditional model that generates 32x32 images in
well under a second on a Colab CPU runtime. The text-to-image stretch goal
runs best on a free T4 runtime.

In [ ]:
# Scoped install for this lab (safe to re-run; preinstalled on Colab).
%pip install -q diffusers transformers accelerate torch matplotlib

In [ ]:
import matplotlib.pyplot as plt
import torch
from diffusers import DDIMScheduler, DDPMPipeline

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Using device:", device)

## 1. Load a pre-trained diffusion pipeline
`google/ddpm-cifar10-32` is a small model trained to generate 32x32 images.
We swap in the DDIM scheduler so we can generate with far fewer steps.

In [ ]:
pipe = DDPMPipeline.from_pretrained("google/ddpm-cifar10-32").to(device)
pipe.scheduler = DDIMScheduler.from_config(pipe.scheduler.config)
print("Pipeline loaded.")

## 2. Generate an image
Generation starts from random noise and denoises it step by step. A fixed seed
makes the result repeatable.

In [ ]:
generator = torch.Generator(device=device).manual_seed(42)

# TODO: call the pipeline with num_inference_steps=25 and the generator,
# then take the first image from the result's .images list.
# Hint: image = pipe(num_inference_steps=25, generator=generator).images[0]
image = ...

plt.imshow(image)
plt.axis("off")
plt.title("Generated from noise (25 steps)")
plt.show()

## 3. More steps, more refinement
Compare the same seed at different step counts. Fewer steps are faster but
rougher; more steps refine the image.

In [ ]:
step_counts = [2, 5, 10, 25, 50]
fig, axes = plt.subplots(1, len(step_counts), figsize=(12, 3))
for ax, steps in zip(axes, step_counts):
    g = torch.Generator(device=device).manual_seed(42)
    # TODO: generate an image with `steps` inference steps and generator `g`,
    # then show it with ax.imshow(...).
    ...
    ax.set_title(f"{steps} steps", fontsize=9)
    ax.axis("off")
plt.show()

## 4. Reproducibility with seeds
The same seed gives the same image; a different seed gives a different one.

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(9, 2.5))
for ax, seed in zip(axes, [0, 1, 2, 3]):
    # TODO: build a generator seeded with `seed`, generate a 25-step image,
    # and show it with ax.imshow(...).
    ...
    ax.set_title(f"seed {seed}", fontsize=9)
    ax.axis("off")
plt.show()

## Stretch goals
1. **Text-to-image.** Run a Stable Diffusion pipeline and generate an image
   from your own prompt. This downloads a larger model and runs much faster on
   a T4 runtime than on CPU. `sd-turbo` needs only one step.
2. **Guidance sweep.** With the text-to-image pipeline, vary `guidance_scale`
   and watch how closely the image follows the prompt.

In [ ]:
# TODO (stretch): load stabilityai/sd-turbo with AutoPipelineForText2Image and
# generate an image from a prompt with num_inference_steps=1, guidance_scale=0.0.

### Expected output
- The first image is a small, colorful 32x32 picture generated from noise.
- At 2 steps the image is rough; by 25-50 steps it is smoother and more
  coherent.
- The same seed always reproduces the same image; different seeds differ.